# Phase 2 — Full Simulation Training (2-Phase Design)

Trains the top-performing agents from Phase 1 (tutorial screening) on the
full stochastic simulation environment.

**Design:** Training is split into two independent phases with checkpoint
persistence. If a crash/restart occurs, Phase 2 can resume from saved
tutorial checkpoints without re-running tutorials.

**Phase 1 — Tutorial Pre-Training:**
- Trains all agent variants through 25 tutorial scenarios (11 tiers)
- 90% mastery gate per tier
- Saves tutorial-complete checkpoints + move CSV logs for the visualizer

**Phase 2 — Curriculum Simulation Training:**
- Loads tutorial checkpoints (can be run independently after kernel restart)
- 11 stages: 20 → 220 imports/day (increment 20)
- 7 simulated days per stage
- Epsilon resets at each stage boundary

**Agents:**
1. Deeper + Munchausen (highest tutorial accuracy)
2. Baseline + Munchausen (fastest to mastery)
3. Residual + Spectral Norm (most consistent)
4. Kitchen Sink (combined agent)

In [1]:
import sys, os, time, random, gc, json
import numpy as np
import torch

# Ensure project root is on path
_cwd = os.getcwd()
if os.path.basename(_cwd) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
elif os.path.isdir(os.path.join(_cwd, "simulation")):
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: /home/franko/CT-DRL-MA
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 2060 SUPER


In [2]:
from simulation.rl.agent_registry import create_agent
from simulation.training.unified_curriculum_trainer import (
    create_env_factory, build_agent_config, CurriculumSchedule,
    CurriculumTrainer,
)
from simulation.training.unified_tutorial_runner import UnifiedTutorialRunner

# ── Shared Configuration ───────────────────────────────────────────
SEED = 42
ROWS, BAYS, TIERS, TRACKS = 5, 58, 5, 7
SPLIT_FACTOR = 20

OUTPUT_BASE = os.path.join(PROJECT_ROOT, "outputs", "phase2")
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Agent configurations: (label, variant_name, backbone_variant)
AGENTS = [
    ("kitchen_sink",        "kitchen_sink",  "kitchen_sink"),
    ("deeper_munchausen",   "munchausen",    "deeper"),
    ("baseline_munchausen", "munchausen",    "baseline"),
    ("residual_specnorm",   "spectral_norm", "residual"),
]

# Tutorial settings
TUTORIAL_EPOCHS = 800
TUTORIAL_MASTERY = 0.9

# Curriculum settings
schedule = CurriculumSchedule(
    start_imports=20,
    increment=20,
    max_imports=220,
    days_per_stage=7,
    epsilon_reset_per_stage=True,
)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def make_agent(variant, backbone):
    """Create an agent via the registry with the correct config."""
    cfg = build_agent_config(
        rows=ROWS, bays=BAYS, tiers=TIERS,
        split_factor=SPLIT_FACTOR, tracks=TRACKS,
    )
    return create_agent(variant, cfg, backbone_variant=backbone), cfg

def ckpt_path(label):
    """Path to the tutorial checkpoint for an agent."""
    return os.path.join(OUTPUT_BASE, label, "checkpoints", "tutorial_complete.pt")

print(f"Output base: {OUTPUT_BASE}")
print(f"Curriculum: {schedule.num_stages} stages, {schedule.days_per_stage} days/stage")
print(f"Agents: {len(AGENTS)}")
for label, variant, backbone in AGENTS:
    print(f"  {label}: variant={variant}, backbone={backbone}")

Output base: /home/franko/CT-DRL-MA/outputs/phase2
Curriculum: 11 stages, 7 days/stage
Agents: 4
  kitchen_sink: variant=kitchen_sink, backbone=kitchen_sink
  deeper_munchausen: variant=munchausen, backbone=deeper
  baseline_munchausen: variant=munchausen, backbone=baseline
  residual_specnorm: variant=spectral_norm, backbone=residual


## Phase 1 — Tutorial Pre-Training

Trains each agent variant through the 25 tutorial scenarios with mastery gates.
Saves checkpoints so Phase 2 can be run independently (even after kernel restart).

In [3]:
tutorial_results = {}

for label, variant, backbone in AGENTS:
    print(f"\n{'='*70}")
    print(f"  Tutorial Training: {label}")
    print(f"{'='*70}\n")
    
    seed_everything()
    
    output_dir = os.path.join(OUTPUT_BASE, label)
    ckpt_dir = os.path.join(output_dir, "checkpoints")
    log_dir = os.path.join(output_dir, "logs")
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    
    # Create agent and environment (with move logging for visualizer)
    agent, cfg = make_agent(variant, backbone)
    params = sum(p.numel() for p in agent.q_net.parameters())
    print(f"  Parameters: {params:,}")
    
    env_factory = create_env_factory(
        rows=ROWS, bays=BAYS, tiers=TIERS,
        tracks=TRACKS, split_factor=SPLIT_FACTOR,
        log_dir=log_dir,
    )
    
    runner = UnifiedTutorialRunner(
        env_factory=env_factory,
        agent_or_config=agent,
        verbose=True,
    )
    
    t0 = time.time()
    summary = runner.train_all(
        epochs=TUTORIAL_EPOCHS,
        mastery_threshold=TUTORIAL_MASTERY,
        window_size=20,
        min_epochs=10,
        log_every=5,
    )
    wall_time = time.time() - t0
    
    # Save checkpoint
    save_path = ckpt_path(label)
    agent.save(save_path)
    print(f"\n  Checkpoint saved: {save_path}")
    
    # Collect results
    tutorial_results[label] = {
        "wall_time_s": wall_time,
        "mastered": summary.get("mastered", False),
        "epochs_completed": summary.get("epochs_completed", 0),
        "current_tier": summary.get("current_tier", "?"),
        "pass_rates": summary.get("pass_rates", {}),
    }
    
    # Print summary
    status = "MASTERED" if summary.get("mastered") else "INCOMPLETE"
    print(f"  {label}: {status} in {wall_time:.0f}s ({wall_time/60:.1f} min)")
    print(f"  Epochs: {summary.get('epochs_completed', '?')}, "
          f"Tier: {summary.get('current_tier', '?')}")
    
    # GPU cleanup
    del runner, agent
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save tutorial summary
summary_path = os.path.join(OUTPUT_BASE, "phase1_tutorial_summary.json")
with open(summary_path, "w") as f:
    json.dump(tutorial_results, f, indent=2, default=str)

print(f"\n{'='*70}")
print(f"Phase 1 complete! Summary: {summary_path}")
print(f"{'='*70}")
for label, info in tutorial_results.items():
    status = "MASTERED" if info["mastered"] else "INCOMPLETE"
    print(f"  {label}: {status} ({info['wall_time_s']:.0f}s)")


  Tutorial Training: kitchen_sink

  Parameters: 177,956

  Tutorial tiers (11 total):
    Tier 0: Primitives [S1, S2, S3, S4]
    Tier 1: Two-action chains [S5, S6]
    Tier 2: Full chains [S7, S8]
    Tier 3: Restack + load [S9, S10]
    Tier 4: Random generalize [S11, S12]
    Tier 5: Multi-step hard [S13, S14]
    Tier 6: Terminal truck [S25, S22, S23, S24]
    Tier 7: Multi-vehicle [S15, S16]
    Tier 8: Bidirectional train [S17]
    Tier 9: Concurrent ops [S18, S19]
    Tier 10: Full complexity [S20, S21]



/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



  Epoch 5/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [====----------------] 20.0%    S3: yard_to_truck
      [================----] 80.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restack + load
    🔒 Tier 4: Random generalize
    🔒 Tier 5: Multi-step hard
    🔒 Tier 6: Terminal truck
    🔒 Tier 7: Multi-vehicle
    🔒 Tier 8: Bidirectional train
    🔒 Tier 9: Concurrent ops
    🔒 Tier 10: Full complexity
    Average (active): 75.0% | Mastered: 2/25 | ε=0.858

  Epoch 10/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [==------------------] 10.0%    S3: yard_to_truck
      [==============------] 70.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restac

/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



  Epoch 5/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [========------------] 40.0%    S3: yard_to_truck
      [========------------] 40.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restack + load
    🔒 Tier 4: Random generalize
    🔒 Tier 5: Multi-step hard
    🔒 Tier 6: Terminal truck
    🔒 Tier 7: Multi-vehicle
    🔒 Tier 8: Bidirectional train
    🔒 Tier 9: Concurrent ops
    🔒 Tier 10: Full complexity
    Average (active): 70.0% | Mastered: 2/25 | ε=0.858

  Epoch 10/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [==============------] 70.0%    S3: yard_to_truck
      [========------------] 40.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restac

/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



  Epoch 5/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [====----------------] 20.0%    S3: yard_to_truck
      [============--------] 60.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restack + load
    🔒 Tier 4: Random generalize
    🔒 Tier 5: Multi-step hard
    🔒 Tier 6: Terminal truck
    🔒 Tier 7: Multi-vehicle
    🔒 Tier 8: Bidirectional train
    🔒 Tier 9: Concurrent ops
    🔒 Tier 10: Full complexity
    Average (active): 70.0% | Mastered: 2/25 | ε=0.858

  Epoch 10/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [========------------] 40.0%    S3: yard_to_truck
      [================----] 80.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restac

KeyboardInterrupt: 

## Phase 2 — Curriculum Simulation Training

Loads tutorial-trained checkpoints and runs the full curriculum (11 stages, 20→220 imports/day).

**This cell can be run independently** — if your kernel was restarted, just re-run
cells 1-2 (setup + config) then skip to here. The tutorial checkpoints in
`outputs/phase2/{label}/checkpoints/tutorial_complete.pt` will be loaded automatically.

In [ ]:
simulation_results = {}

for label, variant, backbone in AGENTS:
    print(f"\n{'='*70}")
    print(f"  Simulation Training: {label}")
    print(f"{'='*70}\n")
    
    # Check tutorial checkpoint exists
    cp = ckpt_path(label)
    if not os.path.exists(cp):
        print(f"  SKIPPING: no tutorial checkpoint at {cp}")
        continue
    
    seed_everything()
    
    output_dir = os.path.join(OUTPUT_BASE, label)
    log_dir = os.path.join(output_dir, "logs")
    
    # Create agent with same architecture, then load tutorial weights
    agent, cfg = make_agent(variant, backbone)
    agent.load(cp)
    print(f"  Loaded tutorial checkpoint: {cp}")
    params = sum(p.numel() for p in agent.q_net.parameters())
    print(f"  Parameters: {params:,}")
    
    env_factory = create_env_factory(
        rows=ROWS, bays=BAYS, tiers=TIERS,
        tracks=TRACKS, split_factor=SPLIT_FACTOR,
        log_dir=log_dir,
    )
    
    t0 = time.time()
    trainer = CurriculumTrainer(
        env_factory=env_factory,
        schedule=schedule,
        agent_config=cfg,
        output_dir=output_dir,
        seed=SEED,
        verbose=True,
        skip_tutorials=True,
        agent=agent,
    )
    
    trainer.train(start_stage=0)
    wall_time = time.time() - t0
    
    simulation_results[label] = {
        "wall_time_s": wall_time,
        "output_dir": output_dir,
    }
    print(f"\n  {label} completed in {wall_time:.0f}s ({wall_time/60:.1f} min)")
    
    # GPU cleanup
    del trainer, agent
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save simulation summary
summary_path = os.path.join(OUTPUT_BASE, "phase2_simulation_summary.json")
with open(summary_path, "w") as f:
    json.dump(simulation_results, f, indent=2, default=str)

print(f"\n{'='*70}")
print("Phase 2 complete!")
print(f"{'='*70}")
for label, info in simulation_results.items():
    print(f"  {label}: {info['wall_time_s']:.0f}s ({info['wall_time_s']/60:.1f} min)")